# Silver Cleaning (A): sifariş axını

**Məsuliyyətli:** Manaf

| Bronze | Silver |
|---|---|
| `bronze.olist_customers_dataset` | `silver.customers` |
| `bronze.olist_orders_dataset` | `silver.orders` |
| `bronze.olist_order_items_dataset` | `silver.order_items` |
| `bronze.olist_order_payments_dataset` | `silver.order_payments` |
| `bronze.api_holidays` | `silver.holidays` |




In [6]:
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")

import pandas as pd
from db import get_engine

In [7]:
import time
from datetime import date, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
from sqlalchemy.engine import URL
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")
from db import get_engine
import os
import json


### 1.1. customers: ilkin yoxlama (bronze)
Bronze-dakı `customers` cədvəlinin vəziyyətini yoxlayırıq: ölçü, tiplər, boş mətn və NULL, tam təkrar və `customer_id` unikallığı. 

- **Tiplər:** bronze-a məlumat xam, olduğu kimi yüklənib, hamısı mətn (`object`) tipindədir. Düzgün tiplər silver-də veriləcək. Burada hazırkı vəziyyəti təsdiqləyirik.
- **Boş mətn və NULL:** çatışmayan dəyər iki formada ola bilər. Bronze-a yükləyəndə boş xanalar `NULL` yox, boş mətn (`""`) kimi saxlanılıb, ona görə ikisini ayrıca yoxlayırıq.

**Nəticə:**
- 99,441 sətir və 5 sütun var. Bütün sütunlar mətn tipindədir.
- Boş mətn, NULL, tam təkrar və `customer_id` təkrarı: hamısı **0**.
- **Yekun:** cədvəl boşluq və təkrar baxımından təmizdir. Bu mərhələdə heç bir sətir silinmir və ya düzəldilmir.

In [8]:
df = pd.read_sql("SELECT * FROM bronze.olist_customers_dataset", get_engine())

print("Ölçü:", df.shape)
print("\nSütunlar və tiplər:")
print(df.dtypes)
print("\nBoş mətn (\"\") sayı:")
print((df == "").sum())
print("\nNULL sayı:")
print(df.isna().sum())
print("\nTam təkrarlanan sətir:", df.duplicated().sum())
print("customer_id təkrarı:", df["customer_id"].duplicated().sum())
print("\nİlk 5 sətir:")
df.head()

Ölçü: (99441, 5)

Sütunlar və tiplər:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object

Boş mətn ("") sayı:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

NULL sayı:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Tam təkrarlanan sətir: 0
customer_id təkrarı: 0

İlk 5 sətir:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### 1.2. customers: `customer_id` və `customer_unique_id` fərqi
Bu iki sütunun eyni şeyi göstərib-göstərmədiyini yoxlayırıq. Hər sütunda neçə fərqli dəyər olduğunu sayırıq, sonra eyni `customer_unique_id`-yə malik sətirləri göstəririk. Bu xana bazada heç nəyi dəyişmir.

**Nəticə:**
- `customer_id`: **99,441** fərqli dəyər (hər sətirdə fərqlidir). `customer_unique_id`: **96,096** fərqli dəyər.
- Fərq **3,345** sətirdir. Bunlar təkrar alış-veriş edən müştərilərin əlavə sifarişləridir.
- Göstərilən nümunələrdə eyni `customer_unique_id` fərqli `customer_id`-lərlə təkrarlanır. Deməli, `customer_id` hər **sifarişə**, `customer_unique_id` isə real **müştəriyə** aiddir.

**Yekun və qərar:**
- Təkrarlanan `customer_unique_id` xəta deyil, **silinmir**.
- Silver-də hər iki sütun saxlanılır: `customer_id` cədvəlləri birləşdirmək (JOIN) üçün, `customer_unique_id` isə müştəri sayı və təkrar alıcı təhlili (məsələn, RFM) üçündür.

In [9]:
print("Fərqli customer_id:", df["customer_id"].nunique())
print("Fərqli customer_unique_id:", df["customer_unique_id"].nunique())

dup = df[df["customer_unique_id"].duplicated(keep=False)].sort_values("customer_unique_id")
dup.head(6)

Fərqli customer_id: 99441
Fərqli customer_unique_id: 96096


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
22066,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
87012,d95f60d70d9ea9a7fe37c53c931940bb,004b45ec5c64187465168251cd1c9c2f,57035,maceio,AL


### 1.3. customers: mətn keyfiyyəti (ştat, zip, şəhər)
Sütunların formatını yoxlayırıq: ştat kodları 2 hərfdirmi, zip kodları 5 simvoldurmu, şəhər adlarında artıq boşluq və ya hərf qarışıqlığı varmı. 


**Nəticə:**
- 27 fərqli ştat kodu var, hamısı 2 simvoldur (99,441 sətirin hamısında).
- Bütün zip kodları 5 simvoldur.
- 4,119 fərqli şəhər var. Adların kənarında artıq boşluq yoxdur və hamısı kiçik hərflə yazılıb.

**Yekun və qərar:**
- Formatla bağlı düzəldiləsi problem tapılmadı, mətn təmizləmə addımı lazım deyil.
- Silver-də `zip_code_prefix` mətn tipində (`CHAR(5)`) saxlanılır, çünki rəqəm olsa başdakı sıfır itər (`01151` → `1151`). Ştat sütunu `CHAR(2)` olur.

In [10]:
print("Ştat sayı:", df["customer_state"].nunique())
print("Ştat uzunluqları:", df["customer_state"].str.len().value_counts().to_dict())
print("Zip uzunluqları:", df["customer_zip_code_prefix"].str.len().value_counts().to_dict())
print("Fərqli şəhər sayı:", df["customer_city"].nunique())
print("Kənarında boşluq olan şəhər:", (df["customer_city"] != df["customer_city"].str.strip()).sum())
print("Kiçik hərflə olmayan şəhər:", (df["customer_city"] != df["customer_city"].str.lower()).sum())

Ştat sayı: 27
Ştat uzunluqları: {2: 99441}
Zip uzunluqları: {5: 99441}
Fərqli şəhər sayı: 4119
Kənarında boşluq olan şəhər: 0
Kiçik hərflə olmayan şəhər: 0


### 1.4. customers: silver-ə yazmaq
Bronze-dakı `customers` cədvəlini oxuyub `silver.customers` kimi yazırıq. Məlumatın özü dəyişmir, çünki yoxlamada düzəldiləsi problem tapılmadı. Fərq budur ki, hər sütuna dəqiq SQL tipi və uzunluq veririk.

- `customer_id`, `customer_unique_id` → `VARCHAR(32)` (ID-lər 32 simvoldur).
- `customer_zip_code_prefix` → `CHAR(5)`. Mətn saxlanılır, rəqəm olsa başdakı sıfır itərdi (`01151` → `1151`).
- `customer_city` → `VARCHAR(100)`, `customer_state` → `CHAR(2)`.
- `if_exists="replace"`: xananı təkrar işlətmək olar, cədvəl yenidən yaranır.

**Nəticə:**
- `silver.customers` cədvəli yarandı. Xananın çıxardığı rəqəm (138) pandas-ın texniki cavabıdır, yazılan sətirlərin sayı deyil. Sətir sayı və tiplər növbəti xanada bazadan oxunaraq təsdiqlənir.
- Bronze toxunulmaz qalır, xam məlumat orada saxlanılır.

In [11]:
from sqlalchemy import types as sqltypes

customers = pd.read_sql("SELECT * FROM bronze.olist_customers_dataset", get_engine())

customers.to_sql(
    "customers",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "customer_id": sqltypes.VARCHAR(32),
        "customer_unique_id": sqltypes.VARCHAR(32),
        "customer_zip_code_prefix": sqltypes.CHAR(5),
        "customer_city": sqltypes.VARCHAR(100),
        "customer_state": sqltypes.CHAR(2),
    },
)

138

In [12]:
print("silver.customers sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.customers", get_engine()).iloc[0, 0])

pd.read_sql("SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'customers'", get_engine())

silver.customers sətir sayı: 99441


,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH
0,customer_id,varchar,32
1,customer_unique_id,varchar,32
2,customer_zip_code_prefix,char,5
3,customer_city,varchar,100
4,customer_state,char,2


**Orders

### 2.1. orders: ilkin yoxlama (bronze)
Bronze-dakı `orders` cədvəlinin vəziyyətini yoxlayırıq: ölçü, boş mətn və NULL, tam təkrar, `order_id` unikallığı və sifariş statusları. Bu xana bazada heç nəyi dəyişmir.


- **Boş mətn və NULL:** bronze-a yükləyəndə boş xanalar `NULL` yox, boş mətn (`""`) kimi saxlanılıb, ona görə ikisini ayrıca yoxlayırıq.
- **Təkrar və `order_id`:** hər sifariş bir dəfə olmalıdır, çünki `order_id` digər cədvəlləri (`order_items`, `order_payments`) birləşdirən açardır.
- **Statuslar:** sifarişin hansı mərhələdə olduğunu göstərir, boş tarixləri başa düşmək üçün lazımdır.

**Nəticə:**
- 99,441 sifariş, 8 sütun. Tam təkrar və `order_id` təkrarı: **0**.
- Boş mətn olan sütunlar: `order_approved_at` (160), `order_delivered_carrier_date` (1,783), `order_delivered_customer_date` (2,965). Digər sütunlarda boşluq yoxdur.
- 8 status var: delivered 96,478, shipped 1,107, canceled 625, unavailable 609, invoiced 314, processing 301, created 5, approved 2 (cəmi 99,441).

**Yekun:** təkrar yoxdur. Boş tarixlər var, amma hələ xəta olduğunu demək olmaz. Hər sifariş eyni mərhələyə çatmır. Onların statusla uyğun olub-olmadığı növbəti xanalarda yoxlanılır.

In [13]:
orders = pd.read_sql("SELECT * FROM bronze.olist_orders_dataset", get_engine())

print("Ölçü:", orders.shape)
print("\nBoş mətn (\"\") sayı:")
print((orders == "").sum())
print("\nNULL sayı:")
print(orders.isna().sum())
print("\nTam təkrar sətir:", orders.duplicated().sum())
print("order_id təkrarı:", orders["order_id"].duplicated().sum())
print("\nSifariş statusları:")
print(orders["order_status"].value_counts())
orders.head()

Ölçü: (99441, 8)

Boş mətn ("") sayı:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

NULL sayı:
order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Tam təkrar sətir: 0
order_id təkrarı: 0

Sifariş statusları:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### 2.2. orders: tarix formatı və boş tarixlərin statusla yoxlanması
İki şeyi yoxlayırıq: (1) tarix sütunları eyni formatdadırmı və təhlükəsiz tarix tipinə çevirmək olarmı, (2) boş tarixlər sifarişin statusuna uyğundurmu. Bu xana bazada heç nəyi dəyişmir.

**Niyə bu yoxlamalar?**
- Tarixlər bronze-da mətn kimi saxlanılıb. Çevirməzdən əvvəl hamısının `YYYY-MM-DD HH:MM:SS` formatında olduğuna əmin olmalıyıq, yoxsa çevirmə səhv verər.
- Boş tarix hər zaman səhv deyil: hələ çatdırılmamış sifarişin çatdırılma tarixi boş olmalıdır. Səhv yalnız statusla uyğun gəlməyəndə olur (məsələn, `delivered` sifarişin çatdırılma tarixi boşdursa).

- 5 tarix sütununun hamısında tanınmayan format: **0**. Boş saylar: təsdiq tarixi 160, daşıyıcıya verilmə 1,783, çatdırılma 2,965.
- Boş tarixlərin çoxu statusla uyğundur: `shipped` sifarişlərdə çatdırılma tarixi yoxdur, `invoiced`, `processing` və `unavailable` sifarişlərdə daşıyıcı və çatdırılma tarixi yoxdur, yeni yaranan (`created`) sifarişlərdə üç tarix də boşdur.
- **Anomaliyalar:** `delivered` sifarişlərdə 14 təsdiq, 2 daşıyıcı və 8 çatdırılma tarixi boşdur. Bundan başqa, 6 ləğv olunmuş (`canceled`) sifarişin çatdırılma tarixi var.

**Yekun və qərar:**
- Tarixləri təhlükəsiz tarix tipinə çevirə bilərik.
- Anomaliyalı sətirlər **silinmir** (silsək, `order_items` və `order_payments`-də sahibsiz sətirlər qalardı), boş tarixlər də **uydurulmur**. Çatdırılma müddəti hesablananda yalnız iki tarixi olan sətirlər götürüləcək.

In [14]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print("Tarix formatı yoxlaması:")
for c in date_cols:
    empty = (orders[c] == "").sum()
    parsed = pd.to_datetime(orders[c].mask(orders[c] == ""), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    bad = parsed.isna().sum() - empty
    print(f"  {c} | boş: {empty} | tanınmayan format: {bad}")

print("\nStatus üzrə boş tarixlər:")
check = orders.assign(
    approved_empty=(orders["order_approved_at"] == ""),
    carrier_empty=(orders["order_delivered_carrier_date"] == ""),
    customer_empty=(orders["order_delivered_customer_date"] == ""),
)
print(check.groupby("order_status")[["approved_empty", "carrier_empty", "customer_empty"]].sum())

Tarix formatı yoxlaması:
  order_purchase_timestamp | boş: 0 | tanınmayan format: 0
  order_approved_at | boş: 160 | tanınmayan format: 0
  order_delivered_carrier_date | boş: 1783 | tanınmayan format: 0
  order_delivered_customer_date | boş: 2965 | tanınmayan format: 0
  order_estimated_delivery_date | boş: 0 | tanınmayan format: 0

Status üzrə boş tarixlər:
              approved_empty  carrier_empty  customer_empty
order_status                                               
approved                   0              2               2
canceled                 141            550             619
created                    5              5               5
delivered                 14              2               8
invoiced                   0            314             314
processing                 0            301             301
shipped                    0              0            1107
unavailable                0            609             609


In [15]:
from sqlalchemy import types as sqltypes
from sqlalchemy.dialects.mssql import DATETIME2

orders = pd.read_sql("SELECT * FROM bronze.olist_orders_dataset", get_engine())

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# boş mətn -> NULL, sonra tarix tipinə
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c].mask(orders[c] == ""), format="%Y-%m-%d %H:%M:%S")

orders.to_sql(
    "orders",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "customer_id": sqltypes.VARCHAR(32),
        "order_status": sqltypes.VARCHAR(20),
        **{c: DATETIME2(precision=0) for c in date_cols},
    },
)

143

### 2.4. orders: silver-i təsdiqləmək
Yazılan `silver.orders` cədvəlini bazadan geri oxuyub yoxlayırıq: sətir sayı bronze ilə eyni qalıbmı və boş mətnlər itkisiz həqiqi `NULL`-a çevrilibmi. Bu xana bazada heç nəyi dəyişmir.

**Niyə bu yoxlama?** Silver-ə yazanda boş mətnləri `NULL`-a çevirdik və tarixləri tarix tipinə saldıq. Çevirmə zamanı sətir itməməli və boşluqların sayı dəyişməməlidir. Bunu bazadan oxuyub təsdiqləyirik.

- `silver.orders`: **99,441** sətir (bronze ilə eyni, heç bir sətir itməyib).
- `NULL` sayları: təsdiq tarixi **160**, daşıyıcıya verilmə tarixi **1,783**, çatdırılma tarixi **2,965**. Bunlar bronze-dakı boş mətn saylarına bərabərdir.

**Yekun:** boş mətnlər itkisiz həqiqi `NULL`-a çevrilib. `orders` cədvəli silver-də hazırdır. 

In [16]:
print(pd.read_sql("SELECT COUNT(*) AS n FROM silver.orders", get_engine()))

pd.read_sql("""
SELECT
  SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS approved_null,
  SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS carrier_null,
  SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS customer_null
FROM silver.orders
""", get_engine())


       n
0  99441


,approved_null,carrier_null,customer_null
0,160,1783,2965


**Items


### 3.1. order_items: ilkin yoxlama (bronze)
Bronze-dakı `order_items` cədvəlinin vəziyyətini yoxlayırıq: boş mətn, təkrar, qiymət və tarix sütunlarının formatı, qiymət aralığı və hər sətrin `silver.orders`-dəki sifarişə aid olub-olmaması. 


- **Cüt açar:** bir sifarişdə bir neçə məhsul ola bilər, ona görə unikallığı `order_id` ilə deyil, `(order_id, order_item_id)` cütü ilə yoxlayırıq.
- **Rəqəm və tarix formatı:** `price`, `freight_value` və `shipping_limit_date` bronze-da mətn kimi saxlanılıb. Çevirməzdən əvvəl hamısının çevrilə bildiyinə əmin olmalıyıq.
- **Sahibsiz sətirlər:** `silver.orders`-də olmayan `order_id`-si olan sətir sonrakı birləşdirmədə itər.

- 112,650 sətir, 7 sütun. Boş mətn, tam təkrar və `(order_id, order_item_id)` təkrarı: **0**.
- `price` və `freight_value` rəqəmə çevrilir (oxunmayan: 0). `price`: 0.85 – 6,735.00, `freight_value`: 0.00 – 409.68. `shipping_limit_date` formatı düzgündür.
- `silver.orders`-də olmayan `order_id`: **0**, sahibsiz sətir yoxdur.

**Yekun və qərar:**
- Cədvəl təmizdir, heç bir sətir silinmir.
- Qeyri-adi görünə bilən dəyərlər (qiymət 6,735.00, daşıma haqqı 0.00) real ola bilər, ona görə saxlanılır və EDA mərhələsində baxılacaq.

In [17]:
items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset", get_engine())

print("Ölçü:", items.shape)
print("Boş mətn sayı:\n", (items == "").sum().to_string())
print("Tam təkrar sətir:", items.duplicated().sum())
print("(order_id, order_item_id) təkrarı:", items.duplicated(subset=["order_id", "order_item_id"]).sum())

price = pd.to_numeric(items["price"], errors="coerce")
freight = pd.to_numeric(items["freight_value"], errors="coerce")
print("price oxunmayan:", price.isna().sum(), "| freight oxunmayan:", freight.isna().sum())
print("price min/max:", price.min(), price.max())
print("freight min/max:", freight.min(), freight.max())

ship = pd.to_datetime(items["shipping_limit_date"].mask(items["shipping_limit_date"] == ""), format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("shipping_limit_date tanınmayan:", ship.isna().sum())

orders_ids = pd.read_sql("SELECT order_id FROM silver.orders", get_engine())["order_id"]
print("silver.orders-də olmayan order_id:", (~items["order_id"].isin(orders_ids)).sum())

Ölçü: (112650, 7)
Boş mətn sayı:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
Tam təkrar sətir: 0
(order_id, order_item_id) təkrarı: 0
price oxunmayan: 0 | freight oxunmayan: 0
price min/max: 0.85 6735.0
freight min/max: 0.0 409.68
shipping_limit_date tanınmayan: 0
silver.orders-də olmayan order_id: 0


In [18]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
from sqlalchemy import types as sqltypes
from sqlalchemy.dialects.mssql import DATETIME2

items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset", get_engine())

items["order_item_id"] = pd.to_numeric(items["order_item_id"])
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S")
items["price"] = pd.to_numeric(items["price"])
items["freight_value"] = pd.to_numeric(items["freight_value"])

items.to_sql(
    "order_items",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "order_item_id": sqltypes.SMALLINT(),
        "product_id": sqltypes.VARCHAR(32),
        "seller_id": sqltypes.VARCHAR(32),
        "shipping_limit_date": DATETIME2(precision=0),
        "price": sqltypes.DECIMAL(10, 2),
        "freight_value": sqltypes.DECIMAL(10, 2),
    },
)

226

### 3.3. order_items: silver-i təsdiqləmək
Yazılan `silver.order_items` cədvəlini bazadan geri oxuyub yoxlayırıq: sətir sayı bronze ilə eyni qalıbmı, pul məbləğləri yazılarkən itməyibmi və sütun tipləri planlaşdırdığımız kimidirmi.


- Pul sütunlarını `DECIMAL(10,2)` tipinə saldıq. Yazılarkən yuvarlaqlaşma və ya itki olmadığını bazadakı cəmi pandas-dakı cəmlə müqayisə edərək təsdiqləyirik.
- Sətir sayı və tiplər yazma xanasının düzgün işlədiyini göstərir.


- `silver.order_items`: **112,650** sətir (bronze ilə eyni).
- `price` cəmi: **13,591,643.70**, `freight_value` cəmi: **2,251,909.54**. SQL və pandas cəmləri eynidir, məbləğlər itməyib.
- Tiplər: `order_id`, `product_id`, `seller_id` → `varchar(32)`, `order_item_id` → `smallint`, `shipping_limit_date` → `datetime2`, `price` və `freight_value` → `decimal(10,2)`.

**Yekun:** `order_items` cədvəli silver-də hazırdır, sətir və məbləğ itkisi yoxdur. Növbəti cədvəl: `order_payments`.

In [20]:
print("Sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.order_items", get_engine()).iloc[0, 0])

sql_sum = pd.read_sql("SELECT SUM(price) AS p, SUM(freight_value) AS f FROM silver.order_items", get_engine())
print("SQL cəmi (price, freight):", sql_sum.iloc[0].tolist())
print("Pandas cəmi (price, freight):", round(items["price"].sum(), 2), round(items["freight_value"].sum(), 2))

pd.read_sql("""
SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'order_items'
ORDER BY ORDINAL_POSITION
""", get_engine())

Sətir sayı: 112650
SQL cəmi (price, freight): [13591643.7, 2251909.54]
Pandas cəmi (price, freight): 13591643.7 2251909.54


,COLUMN_NAME,DATA_TYPE,NUMERIC_PRECISION,NUMERIC_SCALE,CHARACTER_MAXIMUM_LENGTH
0,order_id,varchar,NaN,NaN,32.0
1,order_item_id,smallint,5.0,0.0,NaN
2,product_id,varchar,NaN,NaN,32.0
3,seller_id,varchar,NaN,NaN,32.0
4,shipping_limit_date,datetime2,NaN,NaN,NaN
5,price,decimal,10.0,2.0,NaN
6,freight_value,decimal,10.0,2.0,NaN


### 4.1. order_payments: ilkin yoxlama (bronze)
Bronze-dakı `order_payments` cədvəlinin vəziyyətini yoxlayırıq: boş mətn, təkrar, ödəniş növləri, rəqəm sütunlarının aralığı və hər sətrin `silver.orders`-dəki sifarişə aid olub-olmaması.


- **Cüt açar:** bir sifariş bir neçə ödənişlə ödənilə bilər, ona görə unikallığı `(order_id, payment_sequential)` cütü ilə yoxlayırıq.
- **Rəqəm sütunları:** `payment_sequential`, `payment_installments` və `payment_value` bronze-da mətn kimi saxlanılıb. Çevirməzdən əvvəl hamısının çevrilə bildiyinə və dəyərlərin məntiqli aralıqda olduğuna əmin olmalıyıq.
- **Sahibsiz sətirlər:** `silver.orders`-də olmayan `order_id`-si olan ödəniş sonrakı birləşdirmədə itər.

- 103,886 sətir, 5 sütun. Boş mətn, tam təkrar və `(order_id, payment_sequential)` təkrarı: **0**.
- 5 ödəniş növü: credit_card 76,795, boleto 19,784, voucher 5,775, debit_card 1,529, not_defined 3 (cəmi 103,886).
- Rəqəm sütunları hamısı rəqəmə çevrilir (oxunmayan: 0). Taksit sayı: 0 – 24, məbləğ: 0.00 – 13,664.08.
- `silver.orders`-də olmayan `order_id`: **0**.
- **Qeyri-adi dəyərlər:** məbləği 0 olan 9 sətir, taksit sayı 0 olan 2 sətir və növü `not_defined` olan 3 sətir var.

**Yekun:** cədvəl təmizdir, amma bir neçə qeyri-adi sətir var. Onların hansı sətirlər olduğuna və nə edəcəyimizə növbəti xanada baxılır.

In [21]:
pay = pd.read_sql("SELECT * FROM bronze.olist_order_payments_dataset", get_engine())

print("Ölçü:", pay.shape)
print("Boş mətn sayı:\n", (pay == "").sum().to_string())
print("Tam təkrar sətir:", pay.duplicated().sum())
print("(order_id, payment_sequential) təkrarı:", pay.duplicated(subset=["order_id", "payment_sequential"]).sum())

print("\nÖdəniş növləri:")
print(pay["payment_type"].value_counts().to_string())

seq = pd.to_numeric(pay["payment_sequential"], errors="coerce")
inst = pd.to_numeric(pay["payment_installments"], errors="coerce")
val = pd.to_numeric(pay["payment_value"], errors="coerce")
print("\nOxunmayan -> sequential:", seq.isna().sum(), "| installments:", inst.isna().sum(), "| value:", val.isna().sum())
print("installments min/max:", inst.min(), inst.max())
print("value min/max:", val.min(), val.max())
print("value = 0 olan sətir:", (val == 0).sum())
print("installments = 0 olan sətir:", (inst == 0).sum())

orders_ids = pd.read_sql("SELECT order_id FROM silver.orders", get_engine())["order_id"]
print("silver.orders-də olmayan order_id:", (~pay["order_id"].isin(orders_ids)).sum())


Ölçü: (103886, 5)
Boş mətn sayı:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
Tam təkrar sətir: 0
(order_id, payment_sequential) təkrarı: 0

Ödəniş növləri:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3

Oxunmayan -> sequential: 0 | installments: 0 | value: 0
installments min/max: 0 24
value min/max: 0.0 13664.08
value = 0 olan sətir: 9
installments = 0 olan sətir: 2
silver.orders-də olmayan order_id: 0


### 4.2. order_payments: qeyri-adi sətirlərin yoxlanması
4.1-də tapdıq ki, bəzi ödənişlərin məbləği və ya taksit sayı 0-dır və 3 ödənişin növü `not_defined`-dir. Bu xana həmin sətirləri göstərir ki, nə olduqlarını görək. 

 Qeyri-adi dəyərləri silməzdən və ya düzəltməzdən əvvəl onların nə olduğunu bilməliyik. Hər sətir real sifarişə aiddir və silmək sonrakı birləşdirmələri poza bilər.

** cəmi **11** qeyri-adi sətir var (3 `not_defined` sətrin hamısında məbləğ də 0-dır, ona görə sayılar üst-üstə düşür):
- **6 voucher** ödənişi, məbləğ 0.00. Çox ödənişli sifarişlərin tərkibindədir (ödəniş sıra nömrələri 3, 4, 4, 4, 13, 14).
- **3 `not_defined`** ödənişi, məbləğ 0.00, hamısı sifarişin 1-ci ödənişidir.
- **2 kart ödənişi**, taksit sayı 0, məbləğlər normaldır (58.69 və 129.94).

- Bu sətirlər **saxlanılır**. Hər biri real sifarişə aiddir, sıfır məbləğlər cəmləri dəyişmir və silmək sifariş ilə ödəniş uyğunluğunu poza bilər.
- Sıfır məbləğin və sıfır taksitin səbəbi araşdırılmayıb. Taksit üzrə təhlildə (EDA) bunu nəzərə almaq lazımdır.

In [22]:
odd = pay[(val == 0) | (inst == 0) | (pay["payment_type"] == "not_defined")]
print("Qeyri-adi sətir sayı:", len(odd))
odd

Qeyri-adi sətir sayı: 11


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


In [23]:
from sqlalchemy import types as sqltypes

pay = pd.read_sql("SELECT * FROM bronze.olist_order_payments_dataset", get_engine())

pay["payment_sequential"] = pd.to_numeric(pay["payment_sequential"])
pay["payment_installments"] = pd.to_numeric(pay["payment_installments"])
pay["payment_value"] = pd.to_numeric(pay["payment_value"])

pay.to_sql(
    "order_payments",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "payment_sequential": sqltypes.SMALLINT(),
        "payment_type": sqltypes.VARCHAR(20),
        "payment_installments": sqltypes.SMALLINT(),
        "payment_value": sqltypes.DECIMAL(10, 2),
    },
)

393

In [24]:
print("Sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.order_payments", get_engine()).iloc[0, 0])
sql_sum = pd.read_sql("SELECT SUM(payment_value) AS v FROM silver.order_payments", get_engine()).iloc[0, 0]
print("SQL cəmi:", sql_sum, "| Pandas cəmi:", round(pay["payment_value"].sum(), 2))

pd.read_sql("""
SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'order_payments'
ORDER BY ORDINAL_POSITION
""", get_engine())

Sətir sayı: 103886
SQL cəmi: 16008872.12 | Pandas cəmi: 16008872.12


,COLUMN_NAME,DATA_TYPE,NUMERIC_PRECISION,NUMERIC_SCALE,CHARACTER_MAXIMUM_LENGTH
0,order_id,varchar,NaN,NaN,32.0
1,payment_sequential,smallint,5.0,0.0,NaN
2,payment_type,varchar,NaN,NaN,20.0
3,payment_installments,smallint,5.0,0.0,NaN
4,payment_value,decimal,10.0,2.0,NaN


### 5.1. holidays: ilkin yoxlama (bronze)
Bronze-dakı `api_holidays` cədvəlinin (Braziliya bayramları, API-dan gəlir) vəziyyətini yoxlayırıq: ölçü, sütunlar, boş mətn və NULL, tam təkrar və ilk 10 sətir. Bu cədvəlin sütunlarını əvvəlcədən bilmədiyimiz üçün əvvəl ona baxırıq.


- **Sütunlar:** API-dan gələn cədvəlin quruluşu CSV-lərdən fərqlidir, hansı sütunlar olduğunu və hansının lazım olduğunu görməliyik.
- **Boş mətn və NULL:** API cavabı JSON formatındadır, çatışmayan dəyərlər boş mətn yox, həqiqi `NULL` kimi gəlir.

- 42 sətir, 9 sütun: `date`, `localName`, `name`, `countryCode`, `fixed`, `global`, `counties`, `launchYear`, `types`.
- Boş mətn və tam təkrar: **0**.
- `NULL`: `launchYear` hamısında (42) boşdur, `counties` 39 sətirdə boşdur.
- İlk 10 sətir 2016-cı ilin bayramlarıdır. Çoxu bütün ölkə üçündür, bir bayram isə yalnız São Paulo ştatı üçündür (`["BR-SP"]`).

**Yekun:** cədvəl təmizdir, amma bəzi sütunlar işimizə yaramır (`launchYear` tamamilə boşdur). Hansı sütunların atılacağına növbəti xanada (tarix aralığı və sütun dəyərləri) qərar veririk.

In [25]:
hol = pd.read_sql("SELECT * FROM bronze.api_holidays", get_engine())

print("Ölçü:", hol.shape)
print("\nSütunlar:", hol.columns.tolist())
print("\nBoş mətn sayı:\n", (hol == "").sum().to_string())
print("\nNULL sayı:\n", hol.isna().sum().to_string())
print("\nTam təkrar sətir:", hol.duplicated().sum())
hol.head(10)


Ölçü: (42, 9)

Sütunlar: ['date', 'localName', 'name', 'countryCode', 'fixed', 'global', 'counties', 'launchYear', 'types']

Boş mətn sayı:
 date           0
localName      0
name           0
countryCode    0
fixed          0
global         0
counties       0
launchYear     0
types          0

NULL sayı:
 date            0
localName       0
name            0
countryCode     0
fixed           0
global          0
counties       39
launchYear     42
types           0

Tam təkrar sətir: 0


,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2016-01-01,Confraternização Universal,New Year's Day,BR,False,True,None,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,BR,False,True,None,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,BR,False,True,None,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,BR,False,True,None,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,BR,False,True,None,None,"[""Public""]"
5,2016-04-21,Dia de Tiradentes,Tiradentes,BR,False,True,None,None,"[""Public""]"
6,2016-05-01,Dia do Trabalhador,Labour Day,BR,False,True,None,None,"[""Public""]"
7,2016-05-26,Corpus Christi,Corpus Christi,BR,False,True,None,None,"[""Public""]"
8,2016-07-09,Revolução Constitucionalista de 1932,Constitutionalist Revolution of 1932,BR,False,False,"[""BR-SP""]",None,"[""Public""]"
9,2016-09-07,Dia da Independência,Independence Day,BR,False,True,None,None,"[""Public""]"


### 5.2. holidays: tarix aralığı, təkrar və sütun dəyərləri
5.1-dəki ilkin baxışı davam etdiririk: bayramların tarix aralığını və illər üzrə sayını, eyni tarixin təkrarlanıb-təkrarlanmadığını, `countryCode`, `fixed` və `global` sütunlarının dəyərlərini və regional bayramları yoxlayırıq.


- **Tarix aralığı və illər üzrə say:** API-nın bütün illəri tam gətirdiyinə və bayramların sifariş tarixlərini əhatə edə biləcəyinə əmin olmaq üçün.
- **`countryCode`, `fixed`, `global`:** hansı sütunların faydalı məlumat daşıdığını, hansının hamısında eyni dəyər olduğunu görmək üçün.
- **Regional bayramlar:** bütün ölkə üçün olmayan bayramları tapmaq üçün. Onlar sonrakı birləşdirmədə fərqli davranılmalıdır.

- Tarix aralığı: 2016-01-01 → 2018-12-25. Hər il 14 bayram var (cəmi 42). Eyni tarixi təkrarlanan sətir: **0**.
- `countryCode` hamısında `BR`, `fixed` hamısında `False`: bu iki sütun ayırd edici məlumat daşımır.
- `global`: 39 milli, 3 regional bayram. Regional olanlar hər il 9 iyuldur (`Constitutionalist Revolution of 1932`, yalnız São Paulo, `BR-SP`).

**Yekun və qərarlar:**
- `countryCode`, `fixed` (hamısında eyni dəyər) və `launchYear` (hamısı boşdur) silver-də **atılacaq**.
- Regional bayram yalnız həmin ştatın müştərilərinə tətbiq olunmalıdır. Bunu sonrakı birləşdirmədə nəzərə alacağıq.
- Bayramların sifariş tarixlərini tam əhatə edib-etmədiyi sonra sifariş tarixləri ilə müqayisədə yoxlanılacaq.

In [26]:
print("Tarix aralığı:", hol["date"].min(), "→", hol["date"].max())
print("İl üzrə say:", hol["date"].astype(str).str[:4].value_counts().sort_index().to_dict())
print("Eyni tarixi təkrarlanan sətir:", hol["date"].duplicated().sum())

for c in ["countryCode", "fixed", "global"]:
    print(f"{c}:", hol[c].value_counts().to_dict())

print("\nRegional bayramlar:")
print(hol[hol["global"].astype(str) == "False"][["date", "name", "counties"]].to_string())

Tarix aralığı: 2016-01-01 → 2018-12-25
İl üzrə say: {'2016': 14, '2017': 14, '2018': 14}
Eyni tarixi təkrarlanan sətir: 0
countryCode: {'BR': 42}
fixed: {False: 42}
global: {True: 39, False: 3}

Regional bayramlar:
          date                                  name   counties
8   2016-07-09  Constitutionalist Revolution of 1932  ["BR-SP"]
22  2017-07-09  Constitutionalist Revolution of 1932  ["BR-SP"]
36  2018-07-09  Constitutionalist Revolution of 1932  ["BR-SP"]


In [27]:
from sqlalchemy import types as sqltypes

hol = pd.read_sql("SELECT * FROM bronze.api_holidays", get_engine())

silver_hol = pd.DataFrame({
    "holiday_date": pd.to_datetime(hol["date"].astype(str)).dt.date,
    "local_name": hol["localName"],
    "name": hol["name"],
    "is_national": hol["global"].astype(str).str.lower().eq("true"),
    "state_code": hol["counties"].str.extract(r"BR-([A-Z]{2})")[0],
    "types": hol["types"],
})

silver_hol.to_sql(
    "holidays",
    get_engine(),
    schema="silver",
    if_exists="replace",
    index=False,
    dtype={
        "holiday_date": sqltypes.DATE(),
        "local_name": sqltypes.NVARCHAR(100),
        "name": sqltypes.VARCHAR(100),
        "is_national": sqltypes.Boolean(),
        "state_code": sqltypes.CHAR(2),
        "types": sqltypes.VARCHAR(50),
    },
)

42

### 5.4. holidays: silver-i təsdiqləmək
Yazılan `silver.holidays` cədvəlini bazadan geri oxuyub yoxlayırıq: sətir sayı bronze ilə eyni qalıbmı, Portuqal hərfləri itməyibmi, regional bayramlar düzgün işarələnibmi və sütun tipləri planlaşdırdığımız kimidirmi. 

- `local_name` sütununda Portuqal xüsusi hərfləri (`ç`, `ã`) var. Onları `NVARCHAR` tipi ilə saxladıq, yazılarkən itmədiyini göz ilə yoxlayırıq.
- `global` və `counties` sütunlarını `is_national` və `state_code`-a çevirdik. Regional bayramların düzgün çevrildiyini yoxlayırıq.

- `silver.holidays`: **42** sətir (bronze ilə eyni).
- İlk sətirlərdə `Confraternização Universal` kimi adlar düzgün yazılıb, hərflər itməyib. Milli bayramlarda `state_code` boşdur (`NULL`).
- Regional bayramlar: **3** sətir (2016, 2017 və 2018-ci illərin 9 iyulu), hamısında `is_national = 0` və `state_code = SP`.
- Tiplər: `holiday_date` → `date`, `local_name` → `nvarchar(100)`, `name` → `varchar(100)`, `is_national` → `bit`, `state_code` → `char(2)`, `types` → `varchar(50)`.


In [28]:
print("Sətir sayı:", pd.read_sql("SELECT COUNT(*) AS n FROM silver.holidays", get_engine()).iloc[0, 0])
print(pd.read_sql("SELECT TOP 3 holiday_date, local_name, is_national, state_code FROM silver.holidays ORDER BY holiday_date", get_engine()))
print(pd.read_sql("SELECT holiday_date, name, is_national, state_code FROM silver.holidays WHERE is_national = 0", get_engine()))

pd.read_sql("""
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = 'holidays'
ORDER BY ORDINAL_POSITION
""", get_engine())

Sətir sayı: 42
  holiday_date                  local_name  is_national state_code
0   2016-01-01  Confraternização Universal         True       None
1   2016-02-08                    Carnaval         True       None
2   2016-02-09                    Carnaval         True       None
  holiday_date                                  name  is_national state_code
0   2016-07-09  Constitutionalist Revolution of 1932        False         SP
1   2017-07-09  Constitutionalist Revolution of 1932        False         SP
2   2018-07-09  Constitutionalist Revolution of 1932        False         SP


,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH
0,holiday_date,date,NaN
1,local_name,nvarchar,100.0
2,name,varchar,100.0
3,is_national,bit,NaN
4,state_code,char,2.0
5,types,varchar,50.0


### 6. Silver (A hissəsi): yekun yoxlama
`silver` otağında hazırda hansı cədvəllərin olduğunu və hər birində neçə sətir olduğunu yoxlayırıq.

| Silver cədvəli | Sətir sayı |
|---|---|
| `customers` | 99,441 |
| `holidays` | 42 |
| `order_items` | 112,650 |
| `order_payments` | 103,886 |
| `orders` | 99,441 |

Saylar bronze-dakı saylarla eynidir.

In [29]:
pd.read_sql("""
SELECT t.name AS table_name, SUM(p.rows) AS row_count
FROM sys.tables t
JOIN sys.schemas s ON t.schema_id = s.schema_id
JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0, 1)
WHERE s.name = 'silver'
GROUP BY t.name
ORDER BY t.name
""", get_engine())

,table_name,row_count
0,category_translation,71
1,customers,99441
2,holidays,42
3,order_items,112650
4,order_payments,103886
5,orders,99441
6,products,32951
